In [1]:
import json
import pandas as pd
import re

from bs4 import BeautifulSoup

from langdetect import detect
from langdetect import DetectorFactory

DetectorFactory.seed = 42

with open("../data/raw_ads_2511.json", encoding="utf-8") as f:
    df = pd.DataFrame(json.load(f))

from pathlib import Path

## Datakvalitetskontroll inför städning

In [2]:
print("Antal annonser:", len(df))
print("Unika annons-ID:", df["id"].nunique())

Antal annonser: 6894
Unika annons-ID: 6637


In [3]:
def detect_language(text):
    if not text or not text.strip():
        return "unknown"

    try:
        return detect(text)
    except:
        return "unknown"


# Detektera språk för varje annons
df["language"] = df["description"].apply(detect_language)


# Visa fördelningen av språk
print("Språkfördelning:")
print(df["language"].value_counts())


Språkfördelning:
language
sv         4426
en         2458
unknown       8
fr            2
Name: count, dtype: int64


In [4]:
# Undersöker annonser med språk "unknown"
unknown_ads = df[df["language"] == "unknown"]

for _, row in unknown_ads.iterrows():
    print("=" * 80)
    print("ID:", row["id"])
    print("Titel:", row["title"])
    print("Text:", row["description"])

ID: 29506534
Titel: Teamcenter Solution Architect operations
Text: 
ID: 29737028
Titel: Core Network Analyst
Text: 
ID: 29817808
Titel: Senior Power BI Specialist - Älmhult
Text: 
ID: 30456047
Titel: UI/UX designer
Text: 
ID: 30600803
Titel: Dynamics 365 Architect / D365 Solution Architect  16004
Text: 
ID: 30854632
Titel: Power BI Consultant (Report Development)
Text: 
ID: 30961171
Titel: Senior Data Engineer
Text: 
ID: 30983013
Titel: Linux Engineer
Text: 


In [5]:
# Identifierar annonser som saknar description eller endast innehåller tomrum
missing_description = df[
    df["description"].isna() |
    df["description"].fillna("").str.strip().eq("")
]

print("Antal annonser utan description:", len(missing_description))

missing_description[
    ["id", "title", "employer", "occupation", "description"]
].head(10)

Antal annonser utan description: 8


,id,title,employer,occupation,description
931,29506534,Teamcenter Solution Architect operations,Hitachi Energy Sweden AB,IT-arkitekt/Lösningsarkitekt,
2424,29737028,Core Network Analyst,Semicon Service Nordic AB,Systemanalytiker/Systemutredare,
2824,29817808,Senior Power BI Specialist - Älmhult,Danda AB,"Affärskonsult, IT",
4475,30456047,UI/UX designer,GreenBay GreenTech Nordic AB,IT-utredare,
4903,30600803,Dynamics 365 Architect / D365 Solution Archite...,Veritaz AB,Enterprise Architect,
5455,30854632,Power BI Consultant (Report Development),Semicon Service Nordic AB,"Verksamhetskonsult, IT",
5777,30961171,Senior Data Engineer,Semicon Service Nordic AB,Data scientist,
5825,30983013,Linux Engineer,Ericsson AB,Systemanalytiker/Systemutredare,


In [6]:
# Identifierar alla rader som tillhör ett duplicerat annons-ID
duplicate_ids = df[
    df["id"].duplicated(keep=False)
].sort_values("id")

print("Antal rader som tillhör ett duplicerat ID:", len(duplicate_ids))
print("Antal duplicerade ID:", duplicate_ids["id"].nunique())

duplicate_ids[
    ["id", "version_id", "title", "employer", "published"]
].head(10)


Antal rader som tillhör ett duplicerat ID: 475
Antal duplicerade ID: 218


,id,version_id,title,employer,published
4,29354585,26008c8a15f486a5b9163029fddd5d7fc3c03cb4,Projektledare,Randstad AB,2025-01-02T10:15:14
5,29354585,f0883b87209ad585259381a396b05e5618ea3d93,Projektledare,Randstad AB,2025-01-02T10:15:14
30,29361752,27b784e84eb1a5703befb304f9f76c3a3c8797db,Teknikansvarig VA till SYSTRA AB i Jönköping,SYSTRA AB,2025-01-07T13:27:42
31,29361752,5fddf1db951f3374f7792910c4f425650ce236e6,Teknikansvarig VA till SYSTRA i Jönköping,SYSTRA AB,2025-01-07T13:27:42
37,29362368,98e4dc9c1ec0419c0fcef0319b64909342df680d,Lösningsarkitekt Nätverksinfrastruktur,Försvarsmakten,2025-01-07T15:11:03
38,29362368,aa58e8f5ebbe56dd9dbc558eea70eb14085eb6de,Lösningsarkitekt Nätverksinfrastruktur,Försvarsmakten,2025-01-07T15:11:03
39,29362443,4acb1a3da382edaf553e92b57524ad381d704d85,Lösningsarkitekt IT-säkerhet,Försvarsmakten,2025-01-07T15:28:34
40,29362443,a335ce8a599380486cb101644acedcc27bef5d5d,Lösningsarkitekt IT-säkerhet,Försvarsmakten,2025-01-07T15:28:34
50,29363892,de9e985fb72fb75014b4c4f676ce7c8bee93db9e,Senior Frontend Engineer,Schibsted Sverige AB,2025-01-08T09:56:04
51,29363892,e3e83d3e1186b260000ba39f70d40dbb5265842d,Frontend Engineer,Schibsted Sverige AB,2025-01-08T09:56:04


In [7]:
# Identifierar HTML-taggar som kan utgöra brus i NLP-analysen
html_pattern = r"<[^>]+>"

html_ads = df[
    df["description"]
    .fillna("")
    .str.contains(html_pattern, regex=True)
]

print("Antal annonser med möjlig HTML:", len(html_ads))

html_ads[
    ["id", "title", "description"]
].head(10)

Antal annonser med möjlig HTML: 3


,id,title,description
4717,30542907,Infrastrukturarkitekt till Växjö kommun,Vill du arbeta med teknik som gör verklig skil...
4966,30622455,Chefsarkitekt | Jula | Skara,Är du redo att forma Julas digitala framtid? D...
5607,30911356,Lösningsarkitekt,Vill du vara med och forma framtidens digitala...


In [8]:
# Regex för att identifiera personnummer
personnummer_pattern = r"\b(?:19|20)?\d{6}[-+ ]?\d{4}\b"

personnummer_count = df["description"].fillna("").str.contains(
    personnummer_pattern, regex=True
).sum()

print("Annonser med möjliga personnummer:", personnummer_count)

Annonser med möjliga personnummer: 0


In [9]:
# Regex för att identifiera epost-adresser
email_pattern = r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b"

email_ads = df[
    df["description"]
    .fillna("")
    .str.contains(email_pattern, regex=True)
]

print("Antal annonser med e-post:", len(email_ads))


Antal annonser med e-post: 0


In [10]:
# Regex för att identifiera möjliga svenska telefonnummer
phone_pattern = r"(?<!\d)(?:\+46|0046|0)[\s-]?(?:\d[\s-]?){7,11}(?!\d)"

phone_ads = df[
    df["description"]
    .fillna("")
    .str.contains(phone_pattern, regex=True)
]

print("Antal annonser med möjliga telefonnummer:", len(phone_ads))


Antal annonser med möjliga telefonnummer: 24


In [11]:
# Kontroll av möjliga telefonnummer före rensning
phone_matches = df["description"].fillna("").str.findall(phone_pattern)

phone_results = df.loc[
    phone_matches.str.len() > 0,
    ["id", "title", "employer"]
].copy()

phone_results["phone_match"] = phone_matches[
    phone_matches.str.len() > 0
].apply(lambda x: ", ".join(x))

phone_results

,id,title,employer,phone_match
186,29386902,IT-arkitekt till domänen Business Common,Vattenfall AB,02-02-2025
187,29386904,IT-arkitekt till domänen Business Common,Vattenfall AB,02-02-2025
188,29386905,IT-arkitekt till domänen Business Common,Vattenfall AB,02-02-2025
427,29422415,Senior Kravanalytiker,Deploja AB,03-02-2025\n
535,29439711,"IT-arkitekt, inriktning Digital Arbetsplats",Länsstyrelsen i Västra Götalands län,000-59 000
1045,29533240,Dataanalytiker sökes av Göteborgs Spårvägar,Göteborgs Stads Kollektivtrafik AB,000-52 000
1865,29663343,IT Specialist till VB Energi,AFRY AB,0240-876 00
1866,29663343,IT Specialist till VB Energi,AFRY AB,0240-876 00
2667,29791470,Senior IT-arkitekt,Deploja AB,01-07-2025
3048,29929162,IT Systemförvaltare – Teknikstöd nät och platt...,Försvarsmakten,0589-40400


In [12]:
# Identifierar möjliga telefonnummer i annonstexterna med hjälp av regex
phone_matches = df["description"].fillna("").str.findall(phone_pattern)

# Hämtar de annonser där minst ett möjligt telefonnummer har hittats
phone_ads = df.loc[
    phone_matches.str.len() > 0
].copy()

# Visar telefonträffarna tillsammans med texten runt träffen för manuell granskning
for _, row in phone_ads.iterrows():
    print("=" * 80)
    print("ID:", row["id"])
    print("Titel:", row["title"])
    print("Träff:", phone_matches.loc[row.name])
    
    description = row["description"]
    
    for match in phone_matches.loc[row.name]:
        start = description.find(match)
        context = description[max(0, start - 100): start + len(match) + 100]
        print("Sammanhang:")
        print(context)

ID: 29386902
Titel: IT-arkitekt till domänen Business Common
Träff: ['02-02-2025']
Sammanhang:
 föräldraledighetstillägg med mera. Läs mer om våra förmåner här
. Välkommen med din ansökan senast 02-02-2025. Urval och intervjuer sker löpande. Vi tar endast emot ansökningar via vår webbplats. Vi är övertyg
ID: 29386904
Titel: IT-arkitekt till domänen Business Common
Träff: ['02-02-2025']
Sammanhang:
 föräldraledighetstillägg med mera. Läs mer om våra förmåner här
. Välkommen med din ansökan senast 02-02-2025. Urval och intervjuer sker löpande. Vi tar endast emot ansökningar via vår webbplats. Vi är övertyg
ID: 29386905
Titel: IT-arkitekt till domänen Business Common
Träff: ['02-02-2025']
Sammanhang:
 föräldraledighetstillägg med mera. Läs mer om våra förmåner här
. Välkommen med din ansökan senast 02-02-2025. Urval och intervjuer sker löpande. Vi tar endast emot ansökningar via vår webbplats. Vi är övertyg
ID: 29422415
Titel: Senior Kravanalytiker
Träff: ['03-02-2025\n']
Sammanhang:
n av

## Cleaning

In [13]:
print("Antal annonser före rensning:", len(df))


# Behåll endast svenska annonser
df_clean = df[df["language"] == "sv"].copy()

print("Antal svenska annonser:", len(df_clean))


Antal annonser före rensning: 6894
Antal svenska annonser: 4426


In [14]:
# Ta bort dubbletter baserat på annons-ID
antal_fore_dubblettrensning = len(df_clean)

df_clean = df_clean.drop_duplicates(
    subset="id",
    keep="first"
).copy()

antal_efter_dubblettrensning = len(df_clean)

print("Dubblettrensning:")
print("Antal annonser före dubblettrensning:", antal_fore_dubblettrensning)
print("Antal annonser efter dubblettrensning:", antal_efter_dubblettrensning)
print("Antal borttagna dubbletter:", 
      antal_fore_dubblettrensning - antal_efter_dubblettrensning)

Dubblettrensning:
Antal annonser före dubblettrensning: 4426
Antal annonser efter dubblettrensning: 4266
Antal borttagna dubbletter: 160


In [15]:
# Identifierar annonser som saknar annonstext
missing_description = df_clean[
    df_clean["description"].isna() |
    df_clean["description"].fillna("").str.strip().eq("")
]

print("Annonser utan description efter rensning:", len(missing_description))

Annonser utan description efter rensning: 0


In [16]:
# Kontrollerar om möjliga telefonnummer fortfarande finns kvar efter städning
phone_ads_after_duplicates = df_clean[
    df_clean["description"]
    .fillna("")
    .str.contains(phone_pattern, regex=True)
]

print(
    "Annonser med möjlig telefonträff efter rensning:",
    len(phone_ads_after_duplicates)
)

Annonser med möjlig telefonträff efter rensning: 23


In [17]:
# Ersätter verifierade telefonnummer med neutral markör 
def clean_phone_numbers(text):
    if pd.isna(text):
        return text
    
    return re.sub(phone_pattern, "[PHONE]", text)


df_clean["description"] = df_clean["description"].apply(clean_phone_numbers)


phone_matches_clean = df_clean["description"].fillna("").str.findall(phone_pattern)

print(
    "Antal möjliga telefonträffar efter rensning:",
    phone_matches_clean.str.len().sum()
)

Antal möjliga telefonträffar efter rensning: 0


In [18]:
# Räknar hur många telefonnummer som ersattes med [PHONE]
phone_replacements = df_clean["description"].fillna("").str.count(r"\[PHONE\]").sum()

print("Antal ersatta telefonnummer:", phone_replacements)

Antal ersatta telefonnummer: 23


In [19]:
# Kontrollerar om några HTML-taggar fortfarande finns kvar efter cleaning
html_ads = df_clean[
    df_clean["description"]
    .fillna("")
    .str.contains(html_pattern, regex=True)
]

print("Antal HTML-annonser efter rensning:", len(html_ads))

Antal HTML-annonser efter rensning: 3


In [20]:
# Visar annonser med kvarvarande HTML för manuell kontroll
for _, row in html_ads.iterrows():
    print("=" * 80)
    print("ID:", row["id"])
    print("Titel:", row["title"])
    print(row["description"])

ID: 30542907
Titel: Infrastrukturarkitekt till Växjö kommun
Vill du arbeta med teknik som gör verklig skillnad? Hos oss får du chansen att påverka utvecklingen av en hel kommun, där smarta digitala lösningar bidrar till en enklare, tryggare och mer hållbar vardag för invånare, företag och besökare. Nu söker vi en infrastrukturarkitekt som vill vara en nyckelspelare i vår tekniska utvecklingsresa och som motiveras av att arbeta i ett komplext och samhällsviktigt IT‑landskap. Växjö kommun är Kronobergs största arbetsgivare och varje dag arbetar vi tillsammans för framtidens samhälle. <img src="https://sae3ndp007.blob.core.windows.net/swe/71d22a21-1a8c-479b-8fac-9af5f5be811b.jpg">
 
 
Om rollen 
Som infrastrukturarkitekt blir du en del av vårt team Arkitektur & Utveckling, tillsammans med 12 kollegor inom infrastruktur-, nät- och lösningsarkitektur samt systemutveckling. Här får du en central roll i att forma vår tekniska riktning och skapa robusta och säkra lösningar. Du får arbeta i en 

In [21]:
# Tar bort HTML-taggar men behåller den synliga texten
def clean_html(text):
    if pd.isna(text):
        return text
    
    return BeautifulSoup(text, "html.parser").get_text(" ", strip=True)


df_clean["description"] = df_clean["description"].apply(clean_html)


html_after = df_clean[
    df_clean["description"]
    .fillna("")
    .str.contains(r"<[^>]+>", regex=True)
]

print("Annonser med HTML efter cleaning:", len(html_after))

Annonser med HTML efter cleaning: 0


In [22]:
# Söker efter kontaktrelaterade fraser för att manuellt kontrollera
# om personnamn eller andra personuppgifter förekommer i anslutning till dessa
contact_pattern = (
    r"(?i)(kontaktperson|kontakt|kontakta|ansvarig|rekryterare|"
    r"rekryteringsansvarig|för frågor|vid frågor|har du frågor)"
)

contact_ads = df_clean[
    df_clean["description"]
    .fillna("")
    .str.contains(contact_pattern, regex=True)
]

print("Annonser med kontaktfraser:", len(contact_ads))

for _, row in contact_ads.head(20).iterrows():
    description = row["description"]
    match = re.search(contact_pattern, description)

    print("=" * 80)
    print("ID:", row["id"])
    print("Titel:", row["title"])

    if match:
        start = match.start()
        end = min(len(description), match.end() + 200)

        print(description[start:end].replace("\n", " "))

/tmp/ipykernel_14515/2175045008.py:11: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  .str.contains(contact_pattern, regex=True)


Annonser med kontaktfraser: 1742
ID: 29354585
Titel: Projektledare
kontaktytor och intressanta utmaningar. Då kan denna tjänst vara något för dig. Beskrivning av gruppen, En hel kommun är beroende av att vi genomför vårt uppdrag i allt från naturnära områden till stadsmiljö
ID: 29355065
Titel: Consultant Manager till ett av våra grymma Dynamics 365 F&SCM-team
kontakt och involveras tidigt i affärerna ur ett resurssättningsperspektiv. Viktigt att poängtera är att vi inte jobbar med resurskonsulter, utan att vi är ett projektkonsultföretag vilket gör att dina medar
ID: 29355172
Titel: Systemspecialist med inriktning IT-infrastruktur till Säkra IT-lösningar
kontakter. Önskar du att arbeta delvis på distans? I den här tjänsten kan det finnas möjlighet att arbeta hemifrån upp till 40 procent av din arbetstid. #LI-Hybrid Intervjuer sker via Skype, preliminärt unde
ID: 29355229
Titel: IT arkitekt / Lösningsarkitekt till energisektorn i Umeå
ansvarig för Lösningsarkitekturteamet. Uppdragsperio

- För att identifiera annonsernas språk användes paketet `langdetect`. Endast annonser som klassificerades som svenska behölls för den fortsatta analysen. De annonser som klassificerades som `unknown` granskades manuellt och visade sig bestå av tomma annonser. Dessa ingår därför inte i den fortsatta kompetensanalysen.

- Vid kontroll av potentiella personuppgifter identifierades inga träffar för e-postadresser eller personnummer.  

- Regexbaserad identifiering av telefonnummer gav 23 möjliga träffar. Dessa granskades manuellt för att bedöma om de utgjorde faktiska telefonnummer.   
Några av träffarna visade sig vara telefonnummer till företagsväxlar. Verifierade telefonnummer, inklusive företagsväxlar, ersätts med `[PHONE]` eftersom de inte är relevanta för kompetensanalysen.

- Förekomst av HTML-taggar kontrollerades med regex. Ett fåtal annonser innehöll HTML-formatering, men mängden var begränsad. HTML-taggarna tas bort eftersom de inte innehåller relevant information för kompetensanalysen och annars kan utgöra brus vid den fortsatta NLP-bearbetningen.

- Som en extra kontroll av personnamn söktes kontaktrelaterade fraser i annonstexterna för att undersöka om man kan hitta personnamn. Vid manuell granskning syntes inga direkta personuppgifter, utan i de fall där sådan information verkar ha funnits var den redan ersatt med `****`.     
Det tyder på att personuppgifter redan har maskerats i materialet som hämtats från JobTech Historical API.

- Regexbaserad identifiering kan användas för strukturerade uppgifter som e-postadresser, telefonnummer och personnummer men är mindre lämplig för att identifiera personnamn. Förekomst av personnamn kommer därför undersökas separat med spaCy NER. 



## Identifiering av personnamn med SpaCy

Personuppgifter återfinns vanligen i slutet av annonser i relation till "för fler frågor, kontakta.." eller "rekryterande chef" efterföljt av ett namn. Därför börjar vi med att ladda in texten och tittar på de 300 sista orden i ett urval av annonserna för att se om det finns några personnamn.

In [23]:
print(len(df_clean))

# slumpa 10 annonser och titta på de sista 300 tecknen
sample = df_clean.dropna(subset=["description"]).sample(10, random_state=1)
for _, row in sample.iterrows():
    print(row["id"], "...", row["description"][-300:])
    print("-" * 60)

4266
29928135 ... ta företag och arbetar både nationellt och internationellt. Med baser i Sverige och Spanien erbjuder vi en unik kombination av lokal expertis och global räckvidd. Vårt starka nätverk och djupa branschinsikter gör oss till en självklar partner för företag som vill ligga steget före i sin rekrytering.
------------------------------------------------------------
29692285 ...  abstrahera och förklara komplexa tekniska lösningar på ett lättförståeligt sätt
   Strukturerad och självständig i arbetssättet




Vill du vara med och skapa en lösning som gör verklig skillnad för verksamheten? Välkommen att höra av dig till Soros Consulting med din ansökan eller intresseanmälan.
------------------------------------------------------------
30359337 ... d olika bakgrund. Söker du en tjänst hos oss som omfattas av krigsplacering kommer du få information om det i samband med din anställning. Kommunen har gjort aktiva val i denna rekrytering och vill därför inte bli kontaktade med erb

Utskrifter visar att inga personuppgifter dyker upp i någon av exemplerna. Kan bero på att det antingen inte finns några personuppgifter alls i exempelannonserna (redan borttagna av JobTech) eller att de finns någon annanstans än i slutet av annonstexten. 

### SpaCy pipeline

Skapar en spaCy pipeline där modellen tittar på ett litet urval och på alla entitetstyper, inte bara personer för att identifiera personnamn.

In [24]:
import spacy
from collections import Counter

nlp = spacy.load("sv_core_news_sm", disable=["parser", "lemmatizer"])
print(nlp.get_pipe("ner").labels)       # vilka etiketter finns

sample50 = df_clean.sample(50, random_state=1)

counts = Counter()
per_hits = []                           # (annons-id, träff, var i texten 0-1, omgivande text)

for _, row in sample50.iterrows():
    text = row["description"]
    doc = nlp(text)
    for ent in doc.ents:
        counts[ent.label_] += 1
        if ent.label_ == "PRS":     
            pos = ent.start_char / len(text)
            context = text[max(ent.start_char - 40, 0): ent.end_char + 40].replace("\n", " ")
            per_hits.append((row["id"], ent.text, round(pos, 2), context))

print(counts)
print(len(per_hits), "personträffar")
print(Counter(h[1] for h in per_hits).most_common(30))
for h in per_hits[:30]:
    print(h)

('EVN', 'LOC', 'MSR', 'OBJ', 'ORG', 'PRS', 'TME', 'WRK')
Counter({'LOC': 291, 'ORG': 174, 'PRS': 144, 'TME': 22, 'WRK': 14, 'OBJ': 2, 'MSR': 2, 'EVN': 2})
144 personträffar
[('Kubernetes', 7), ('Castra', 6), ('Van', 4), ('Helsingborgs', 4), ('Sopra Steria', 4), ('Databricks', 3), ('Verifieds', 3), ('workshops', 3), ('Avaron AB', 3), ('Hybrid', 3), ('Data Scientist', 3), ('Enterprise Architecture', 3), ('Erfarenhet', 2), ('Förmåga', 2), ('Manager', 2), ('• Erfarenhet', 2), ('Data Science', 2), ('Ansök', 2), ('API:er', 2), ('Walley', 2), ('LinkedIn', 2), ('Azure Databricks', 2), ('Product Owner', 1), ('Teams', 1), ('Ansvara', 1), ('Excel', 1), ('Vänersborgs', 1), ('Verified Global', 1), ('Contract Lifecycle', 1), ('Team Diamant', 1)]
('29928135', 'Product Owner', 0.0, 'Vi söker en Product Owner inom Communication & Collaboration. Hyb')
('29928135', 'Teams', 0.43, 'ap om Microsoft 365:s samarbetsverktyg (Teams, SharePoint, Outlook, OneDrive m.fl.). ')
('29692285', 'Ansvara', 0.28, 'en Som

### Vad resultatet visar

Av de träffar som syns är ingen ett personnamn, vilket tyder på att det flesta träffar faktiskt är falska träffar. 

Noll riktiga namn bland urvalen kan betyda två olika saker:

1. Namnen finns inte i texten, eftersom JobTech tagit bort dem.
2. Namnen finns, men modellen är för dålig på jobbannonser för att hitta dem.

### Litet test

Testar om modellen över huvud taget kan hitta namn i en annons med hjälp av exempeltext. 

In [25]:
test_names = [
    "Kontakta rekryterande chef Anna Svensson på telefon 070 123 45 67.",
    "Har du frågor om tjänsten? Hör av dig till Erik Lindqvist.",
    "Fackliga företrädare: Maria Johansson och Karl Andersson.",
    "Vid frågor kontakta Fatima Hassan, rekryterare.",
]

base = df_clean["description"].iloc[0]
for sentence in test_names:
    doc = nlp(base[-300:] + " " + sentence)
    print([(e.text, e.label_) for e in doc.ents if e.start_char >= len(base[-300:])])

[('Anna Svensson', 'PRS')]
[('Erik Lindqvist', 'PRS')]
[('Maria Johansson', 'PRS'), ('Karl Andersson', 'PRS')]
[('Fatima Hassan', 'PRS')]


Modellen hittar testnamnen vilket tyder på att modellen bör kunna identifiera personnamn i annonstexterna, även om flest träffar verkar vara falska.

Testar att köra pipelinen på alla annonser för att säkerställa att inga personnamn har missats att rensas bort. 

In [26]:
nlp = spacy.load("sv_core_news_sm", disable=["parser", "lemmatizer"])
print(nlp.get_pipe("ner").labels)       # vilka etiketter finns

counts = Counter()
per_hits = []                           # (annons-id, träff, var i texten 0-1, omgivande text)

for _, row in df_clean.iterrows():
    text = row["description"]
    doc = nlp(text)
    for ent in doc.ents:
        counts[ent.label_] += 1
        if ent.label_ == "PRS":     
            pos = ent.start_char / len(text)
            context = text[max(ent.start_char - 40, 0): ent.end_char + 40].replace("\n", " ")
            per_hits.append((row["id"], ent.text, round(pos, 2), context))

print(counts)
print(len(per_hits), "personträffar")
print(Counter(h[1] for h in per_hits).most_common(30))
for h in per_hits[:30]:
    print(h)

('EVN', 'LOC', 'MSR', 'OBJ', 'ORG', 'PRS', 'TME', 'WRK')
Counter({'LOC': 18075, 'ORG': 13662, 'PRS': 11972, 'TME': 1202, 'WRK': 1143, 'OBJ': 268, 'MSR': 192, 'EVN': 122})
11972 personträffar
[('workshops', 476), ('• Erfarenhet', 437), ('Avaron AB', 379), ('Kubernetes', 264), ('Azure', 218), ('Förmåga', 199), ('Sopra Steria', 193), ('Leda', 179), ('Fellowmind', 147), ('Ansök', 142), ('Intervjuer', 138), ('Enterprise Architect', 115), ('LinkedIn', 105), ('Databricks', 101), ('Castra', 98), ('Java', 93), ('Data Scientist', 88), ('Python', 87), ('Enterprise', 84), ('Erfarenhet', 81), ('Västra Götalandsregionen', 71), ('Trusted Advisor', 63), ('Ansvara', 60), ('Hybrid', 60), ('Kafka', 60), ('Manager', 58), ('Van', 58), ('Delge', 58), ('Sälj', 57), ('Great Place', 56)]
('29354331', 'Acquirer', 0.42, 'nom hela betalcykeln, som ex. Merchant, Acquirer, Issuer, Card schemes och Open Banking.')
('29354331', 'Issuer', 0.43, 'etalcykeln, som ex. Merchant, Acquirer, Issuer, Card schemes och Open Ban

Modellen identifierar 11 972 personträffar, varav majoriteten ser ut att fortfarande vara falska träffar. För att undersöka vidare om det finns några riktiga personnamn så behöver träffarna vidare granskas. 

### Filter för att hitta riktiga personnamn

För att försöka identifiera vad som är riktiga personamn bland alla träffar skapar vi filter som sorterar bort det som med säkerhet inte är ett namn för att minska antalet träffar att manuellt granska. 

- Filter 1: ser det ut som ett namn? Riktiga namn är ett till tre ord som alla börjar med stor bokstav och i övrigt har små bokstäver.
- Filter 2: används ordet som ett vanligt ord? Ett vanligt ord som "workshops" eller "solutions" förekommer ofta med liten bokstav, medan ett förnamn som "Anna" gör det nästan aldrig. Filtret räknar därför hur ofta varje ord förekommer med liten bokstav i alla annonser.
- Filter 3: Står träffen där en kontaktperson brukar stå?

In [27]:
from collections import Counter

# Gör om träfflistan till en tabell
hits = pd.DataFrame(per_hits, columns=["id", "name", "pos", "context"])

# Filter 1: ser det ut som ett namn?
NAME_LIKE = re.compile(r"^[A-ZÅÄÖ][a-zåäö]+(?: [A-ZÅÄÖ][a-zåäö]+){0,2}$")
hits = hits[hits["name"].apply(lambda n: bool(NAME_LIKE.match(n)))]
print(len(hits), "träffar efter strukturfilter")

# Filter 2: används ordet som ett vanligt ord?
lower_counts = Counter()
for text in df_clean["description"]:
    lower_counts.update(re.findall(r"\b[a-zåäö]{2,}\b", text))

def is_common(name, min_count=20):
    return all(lower_counts[w.lower()] >= min_count for w in name.split())

hits["common_word"] = hits["name"].apply(is_common)

skills = json.load(open("../data/skills_2511.json", encoding="utf-8"))
skill_words = {s["skill"].lower() for s in skills}
hits["is_skill"] = hits["name"].str.lower().isin(skill_words)

# Filter 3: står träffen där en kontaktperson brukar stå?
cue = re.compile(r"kontakt|rekryter|frågor|ansvarig|hör av|ring |mejla|maila|fackliga",
                 re.IGNORECASE)
hits["near_cue"] = hits["context"].apply(lambda c: bool(cue.search(c)))
hits["late"] = hits["pos"] > 0.6

# Nivåer att läsa
tier1 = hits[hits["near_cue"] & hits["late"] & ~hits["is_skill"]]
tier2 = hits[~hits["common_word"] & ~hits["is_skill"] & ~hits.index.isin(tier1.index)]
print(len(tier1), "i nivå 1,", len(tier2), "i nivå 2")

9136 träffar efter strukturfilter
429 i nivå 1, 5755 i nivå 2


- 11 972 träffar → 9 136 efter strukturfiltret. En ganska liten del av träffarna såg inte ut som ett namn alls. Det är orden med små bokstäver, versaler (AB), siffror, skiljetecken och fler än tre ord.
- 429 i nivå 1 avser träffar som står nära en kontaktfras och sent i texten, och som inte är en känd kompetens. Det här är den mest sannolika platsen för riktiga namn.
- 5 755 i nivå 2 avser träffar som “inte är ett vanligt ord med liten bokstav” och fångar också produktnamn, bolag och titlar som alltid skrivs med stor bokstav (Azure-liknande ord, Scrum Master, Avaron), eftersom de aldrig förekommer i lowercase och därför inte räknas som vanliga ord. Nivån är alltså inte “troliga namn” utan “det som filtren inte kunde avfärda”.
- Resterande cirka 2 952 avser de träffar som filtret räknade som vanliga ord eller kompetenser och inte låg i nivå 1 och är nästan säkert falska. 

### Plocka ut unika strängar

In [28]:
print(tier1["name"].nunique(), "unika strängar i nivå 1")
print(tier2["name"].nunique(), "unika strängar i nivå 2")

# Kollar nivå 1 efter personnamn: läs alla, grupperat
review1 = (tier1.groupby("name")
                .agg(annonser=("id", "nunique"), exempel=("context", "first"))
                .sort_values("annonser"))
print(review1.to_string())

# utskriften i output har tagits bort pga innehåller personnamn

169 unika strängar i nivå 1
1252 unika strängar i nivå 2
                               annonser                                                                                                        exempel
name                                                                                                                                                  
Advania                               1                        r rekryteringsprocessen i samarbete med Advania, tjänsten är en direktanställning hos A
Altova Mapforce                       1                het av mappningsverktyg, som exempelvis Altova Mapforce Erfarenhet av Java och Spring Boot Erfa
Administrator                         1                  ertifiering ServiceNow Certified System Administrator (CSA) ServiceNow Certified Implementati
Apache                                1                         t av Java och Spring Boot Erfarenhet av Apache Camel Erfarenhet av Kubernetes och Tanz
Ansvara                              

Personnamn går nu att identifiera i nivå 1-listan med unika strängar. Dessa kommer att behöva maskeras, även utskriften av review1 har tagits bort inför commit till Git. 

In [29]:
# Kollar nivå 2 efter personnamn: läs alla, grupperat
# utskriften i output tagits bort pga kan innehålla personnamn
review2 = (tier2.groupby("name")
                .agg(annonser=("id", "nunique"), exempel=("context", "first"))
                .sort_values("annonser"))
print(review2.to_string())

                                       annonser                                                                                                                exempel
name                                                                                                                                                                  
Workday Integration Framework                 1          ompetenskrav Dokumenterad erfarenhet av Workday Integration Framework Workday inbound och outbound-integratio
About Rasulson Consulting                     1              vå av relevant erfarenhet (antal år)    About Rasulson Consulting Rasulson Consulting is a specialized st
About Epical                                  1                           Sverige och ha giltigt arbetstillstånd. About Epical  Epical is a Nordic data consultancy dr
Zeek                                          1                                   ätverkstrafik t.ex. Wireshark, T-Shark, Zeek · Förståelse för internetkommunikation

Inga personnamn identifieras i nivå 2-listan, därför behövs ingen maskering göras. 

In [30]:
# skriver om review1 till vanlig kolumn med radnummer
# utskriften i output har tagits bort pga innehåller personnamn
review1 = review1.reset_index()                    
print(review1[["name", "annonser"]].to_string())    

                              name  annonser
0                          Advania         1
1                  Altova Mapforce         1
2                    Administrator         1
3                           Apache         1
4                          Ansvara         1
5             Ants Tech Recruiters         1
6                          Android         1
7                     Andara Group         1
8                  Azure Solutions         1
9                    Azure Synapse         1
10                          Blazor         1
11                           Bonus         1
12         Awexia Executive Search         1
13          Azure Architect Expert         1
14                     Azure Cloud         1
15                    Apache Kafka         1
16           Briljantin Consulting         1
17                      Databricks         1
18            Databricks Workflows         1
19                 Data Management         1
20                      Danielsson         1
21        

### Maskera hittade personnamn i nivå 2

In [31]:
# radnummer för manuellt identifierade personnamn
real_idx = [20, 26, 27, 37, 39, 50, 61, 106, 155]                # radnumren med personnam
real_names = review1.loc[real_idx, "name"].tolist()
print(len(real_names), "riktiga personnamn")

9 riktiga personnamn


In [32]:
# maskera längst först, så att "Förnamn Efternamn" byts ut före "Förnamn"
names_sorted = sorted(real_names, key=len, reverse=True)
pattern = re.compile(r"\b(?:" + "|".join(re.escape(n) for n in names_sorted) + r")\b")

df_clean["description"] = df_clean["description"].apply(lambda t: pattern.sub("[NAMN]", t))

In [33]:
# kontrollera att maskeringen fungerade
remaining = df_clean["description"].apply(lambda t: bool(pattern.search(t))).sum()
print(remaining, "annonser innehåller fortfarande ett valt namn")   # ska vara 0

masked = df_clean["description"].str.count(r"\[NAMN\]").sum()
print(masked, "namn maskerade i", (df_clean["description"].str.contains(r"\[NAMN\]")).sum(), "annonser")

0 annonser innehåller fortfarande ett valt namn
16 namn maskerade i 16 annonser


### Spara städad df till json

In [34]:
out_path = Path("..") / "data" / "ads_clean.json"
out_path.parent.mkdir(exist_ok=True)                # skapar mappen om den saknas, annars händer inget

df_clean.to_json(out_path, orient="records", force_ascii=False, indent=2)
print("Sparade", len(df_clean), "annonser till", out_path.resolve())

Sparade 4266 annonser till /home/irene/Desktop/10 Projekt i Data Science/End-to-end-projekt-i-Data-Science/data/ads_clean.json
